# 🖨️ Extractor de datos — Hojas de contador de impresora
### Instrucciones
1. Menú → **Entorno de ejecución** → **Cambiar tipo de entorno de ejecución** → selecciona **GPU T4**
2. Ejecuta las celdas en orden con ▶
3. En la celda 3 sube la(s) foto(s) de la hoja de contador
4. En la celda 4 se genera y descarga el Excel

In [ ]:
!pip install -q "transformers>=4.45.0" accelerate qwen-vl-utils Pillow openpyxl

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
import torch

model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")
model.eval()

In [ ]:
from google.colab import files
from PIL import Image, ImageOps
from qwen_vl_utils import process_vision_info
import re, io, torch

CAMPOS = [
    "Serial No.", "Default Password", "Total Page Count",
    "Copy Count", "PC-Print Count", "FAX Count", "Other Count",
    "Toner Cartridge", "Toner Fecha", "Drum Unit", "Laser Unit",
]

PROMPT = """Read this printer counter report image and extract the data below.
Reply ONLY with Field: Value lines, no markdown, no asterisks, no brackets, no extra text.

Serial No.: 
Default Password: 
Total Page Count: 
Copy Count: 
PC-Print Count: 
FAX Count: 
Other Count: 
Toner Cartridge: 
Toner Date: 
Drum Unit: 
Laser Unit: 

Rules:
- Serial No. is an alphanumeric code in the header area of the sheet.
- Default Password is an alphanumeric code near Serial No. It contains a mix of letters and digits (NOT a number-only value like 0301 or 0001).
- Return count values exactly as shown in the image: if the sheet shows X/Y keep X/Y, if it shows only X keep X.
- Toner Cartridge: number from the "Replace Count" section, NOT from "Remaining Life".
- Toner Date: date shown beside the Toner Cartridge value in the "Replace Count" section.
- Write N/A if a field is not visible."""

def preguntar_modelo(imagen):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": imagen},
            {"type": "text",  "text": PROMPT},
        ],
    }]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, padding=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        generated = model.generate(**inputs, max_new_tokens=300, do_sample=False)
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated)]
    return processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

def parsear_respuesta(texto):
    mapa = {
        "Serial No.": "Serial No.", "Default Password": "Default Password",
        "Total Page Count": "Total Page Count", "Copy Count": "Copy Count",
        "PC-Print Count": "PC-Print Count", "FAX Count": "FAX Count",
        "Other Count": "Other Count", "Toner Cartridge": "Toner Cartridge",
        "Toner Date": "Toner Fecha", "Drum Unit": "Drum Unit", "Laser Unit": "Laser Unit",
    }
    resultado = {c: "" for c in CAMPOS}
    for linea in texto.splitlines():
        linea_limpia = re.sub(r'\*+', '', linea).strip()
        for clave, campo in mapa.items():
            if linea_limpia.startswith(clave + ":"):
                valor = linea_limpia.split(":", 1)[1].strip().strip("[]").strip()
                resultado[campo] = "" if valor.upper() == "N/A" else valor
                break
    return resultado

subidos = files.upload()
resultados = []

for nombre_archivo, contenido in subidos.items():
    imagen = ImageOps.exif_transpose(Image.open(io.BytesIO(contenido))).convert("RGB")
    respuesta_raw = preguntar_modelo(imagen)
    print(f"\n{nombre_archivo}\n{respuesta_raw}")
    fila = {"Archivo": nombre_archivo}
    fila.update(parsear_respuesta(respuesta_raw))
    resultados.append(fila)

In [ ]:
from google.colab import files as colab_files
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
import io as _io

wb = Workbook()
ws = wb.active
ws.title = "Contadores"
encabezados = ["Archivo"] + CAMPOS

fill = PatternFill("solid", fgColor="2E75B6")
for col_idx, titulo in enumerate(encabezados, start=1):
    c = ws.cell(row=1, column=col_idx, value=titulo)
    c.fill = fill
    c.font = Font(bold=True, color="FFFFFF")
    c.alignment = Alignment(horizontal="center")

for fila_idx, fila in enumerate(resultados, start=2):
    for col_idx, campo in enumerate(encabezados, start=1):
        ws.cell(row=fila_idx, column=col_idx, value=fila.get(campo, ""))

for col in ws.columns:
    max_len = max((len(str(c.value or "")) for c in col), default=10)
    ws.column_dimensions[col[0].column_letter].width = min(max_len + 4, 40)

buf = _io.BytesIO()
wb.save(buf)
buf.seek(0)
with open("contadores.xlsx", "wb") as f:
    f.write(buf.read())

colab_files.download("contadores.xlsx")